In [1]:
!pip install unsloth trl peft accelerate bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 74.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.9/421.9 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 79.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 85.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 50.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0

In [2]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = 'unsloth/Phi-3-mini-4k-instruct-bnb-4bit',
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.8: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/194 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/458 [00:00<?, ?B/s]

```
**`load_in_4bit = True` means:**
The model weights are **quantized from their original precision (usually float16 or float32) down to 4-bit integers**.
1. **Original weights**: Typically stored as **16-bit floats (FP16)** or **32-bit floats (FP32)** (Each weight = 16 or 32 bits)
2. **4-bit quantization**: Each weight is compressed to **4 bits**

**Example calculation:**
If your model has **3.8 billion parameters** (like Phi-3-mini):
- **FP16 (16-bit)**: 3.8B × 16 bits = **7.6 GB** memory
- **4-bit**: 3.8B × 4 bits = **1.9 GB** memory
You get a 4× memory reduction compared to 16-bit.

**It IS:**
- ✅ **Each individual weight** gets quantized from 16/32 bits → 4 bits
- ✅ ~**75% memory reduction** (16-bit → 4-bit)
- ✅ Slight accuracy loss (acceptable for most tasks)

**The "bitsandbytes" library** (imported in Cell 1) handles this quantization using techniques like **NF4 (NormalFloat4)** to minimize quality loss while

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import json
from datasets import Dataset

with open("/content/drive/MyDrive/AI_Docs/Data/al_bdaya_course_assistant_dataset.json", "r", encoding="utf-8") as f:
    data = json.load(f)

ds = Dataset.from_list(data)

def to_text(ex):
    out = ex.get("output", None)

    if out is None:
        msgs = [
            {"role": "user", "content": ex["input"]},
            {"role": "assistant", "content": ""},
        ]
    else:
        if not isinstance(out, str):
            out = json.dumps(out, ensure_ascii=False)

        msgs = [
            {"role": "user", "content": ex["input"]},
            {"role": "assistant", "content": out},
        ]

    return {
        "text": tokenizer.apply_chat_template(
            msgs,
            tokenize=False,
            add_generation_prompt=False
        )
    }


dataset = ds.map(to_text, remove_columns=ds.column_names)


Map:   0%|          | 0/1100 [00:00<?, ? examples/s]

- `ex` is **a single  dictionary from dataset**
- each `ex` is a dictionary entry passed by the `.map()` function.
- Dataset is a list of dict for example
```json
[
  {
    "input": "What is covered in lecture 19?",
    "output": "Lecture 19 covers advanced topics in..."
  },
  {
    "input": "Explain gradient descent",
    "output": "Gradient descent is an optimization..."
  }
]
```
**What `to_text(ex)` does:**
1. Takes `ex` (one row dict)
2. Extracts `input` and `output`
3. Formats them into a chat template
4. Returns `{"text": "<formatted_chat_template>"}`

**The result:**
dataset transforms from: `[{input: "...", output: "..."}]`
To: `[{text: "<|user|>...<|assistant|>...<|end|>"}]`

### Define LoRA adapters

In [5]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 64,

    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
    lora_alpha = 64 * 2,
    lora_dropout = 0,
    bias = 'none',
    use_gradient_checkpointing = 'unsloth',
)


Unsloth 2026.4.8 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


- Adds small trainable matrices (A, B) to target layers
- Base model weights = frozen (not trained)
- Only adapter weights = trainable
```
r=64 is quite high (common: 8, 16, 32) - you're using a lot of capacity
Higher r = more parameters = more expressiveness but slower/more memory
Common pattern: alpha = 2 × r
lora_dropout = 0 Typical for small datasets or when you want maximum adaptation

_proj = Projection = a linear layer (matrix multiplication), It "projects" the input from one dimension to another.
# A projection is just:
output = input @ W + b  # Matrix multiplication
# or in PyTorch:
nn.Linear(input_dim, output_dim)
```
```
**The Transformer Block Structure**
Each transformer layer has **2 main components**:
**1. Attention Mechanism** (the Q, K, V)
Input (hidden_size)
    ↓
┌─────────────────────────────┐
│  q_proj: Linear(hidden → hidden)  ← Projects to Query
│  k_proj: Linear(hidden → hidden)  ← Projects to Key  
│  v_proj: Linear(hidden → hidden)  ← Projects to Value
└─────────────────────────────┘
    ↓
  Attention(Q, K, V)
    ↓
┌─────────────────────────────┐
│  o_proj: Linear(hidden → hidden)  ← Output projection
└─────────────────────────────┘
- **`q_proj`**: Creates the **Query** matrix from input
- **`k_proj`**: Creates the **Key** matrix from input
- **`v_proj`**: Creates the **Value** matrix from input
- **`o_proj`**: Projects the attention output back to hidden dimension

**2. Feed-Forward Network (MLP/FFN)**
After attention, there's a **2-layer MLP**:
Input (hidden_size = 3072)
    ↓
┌─────────────────────────────────┐
│ gate_proj: Linear(3072 → 8192)  │ ← Gating mechanism (controls information flow)
│ up_proj:   Linear(3072 → 8192)  │ ← Parallel expansion to gate
└─────────────────────────────────┘
    ↓
  Activation (SwiGLU in Phi-3)
    ↓
┌─────────────────────────────────┐
│ down_proj: Linear(8192 → 3072)  │ ← Compress back to original size
└─────────────────────────────────┘
**Why 3 projections here?**
Modern architectures (like Phi-3, LLaMA) use **SwiGLU activation**:
# Traditional FFN (2 layers):
hidden = activation(up_proj(x))
output = down_proj(hidden)

# SwiGLU (3 layers):
gate = gate_proj(x)   # One branch
up = up_proj(x)       # Another branch
hidden = swish(gate) * up  # Element-wise multiply
output = down_proj(hidden)
```
```
## **"ALL major linear layers"**
A transformer block looks like:
┌──────────────────────────┐
│   Input Embedding        │
└──────────────────────────┘
           ↓
┌──────────────────────────┐
│  Attention Block:        │
│   - q_proj ← LoRA here   │
│   - k_proj ← LoRA here   │
│   - v_proj ← LoRA here   │
│   - o_proj ← LoRA here   │
└──────────────────────────┘
           ↓
┌──────────────────────────┐
│  FFN/MLP Block:          │
│   - gate_proj ← LoRA here│
│   - up_proj   ← LoRA here│
│   - down_proj ← LoRA here│
└──────────────────────────┘
           ↓
      (Repeat × N layers)

**"Major linear layers"** = all the big matrix multiplications (7) where most computation happens.
By targeting all 7 projections, you're adapting:
- ✅ How attention computes Q, K, V, and output
- ✅ How the FFN processes information

**Why not target everything?**
You could also target:
- **Layer norms** (usually not worth it)
- **Embeddings** (input/output token embeddings)
- But **linear projections = 95%+ of the model's learnable parameters**

### Train adapter weights on your data

In [7]:
from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    train_dataset = dataset,
    tokenizer = tokenizer,    # For encoding text during training
    dataset_text_field = 'text',   # column contains the training text -  dataset has {"text": "<formatted_chat>"}
    max_seq_length = 2048,      # Maximum tokens per training example
    args = SFTConfig(                             # Training Arguments
        per_device_train_batch_size = 2,    # Process 2 examples per GPU per forward pass - Small because of memory constraints - ✅ Take 2 examples and pass them through the model (forward pass)
        gradient_accumulation_steps = 4,     # Accumulate gradients over 4 mini-batches before updating weights - Effective batch size = 2 × 4 = 8 -- Step 1: Forward pass (batch=2) → compute gradients → DON'T update ..... Step 4: Forward pass (batch=2) → accumulate gradients → UPDATE weights - ✅ Wait for 4 batches (each batch = 2 examples) then update weights
        warmup_steps = 10,   # Learning rate starts at 0 and gradually increases to target LR over 10 steps After step 10, LR stays constant (or follows a schedule) -- Prevents unstable training at the start
        max_steps = 60,    # Train for 60 optimization steps (weight updates) -- With effective batch size = 8: processes 60 × 8 = 480 examples -- ✅ Loop 60 times (60 weight updates)
        logging_steps = 1,    # Log loss/metrics every step - Good for debugging
        output_dir = "outputs",   # Save checkpoints and logs to ./outputs/ folder
        optim = "adamw_8bit",  # Uses 8-bit Adam optimizer (from bitsandbytes) - Memory-efficient - Saves ~50% optimizer memory vs standard AdamW - Perfect for fine-tuning with limited VRAM
        num_train_epochs = 3
    ),
)

trainer.train()


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/1100 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,100 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 119,537,664 of 3,940,617,216 (3.03% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
1,3.059654
2,2.790001
3,3.060430
4,2.805604
5,2.966484
6,2.374962
7,2.509811
8,2.419394
9,2.219056
10,2.374720


Unsloth: Will smartly offload gradients to save VRAM!


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-60/tokenizer_config.json.


tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

Unsloth: Preserved sentencepiece asset `tokenizer.model` in outputs/checkpoint-60.


TrainOutput(global_step=60, training_loss=1.392977664868037, metrics={'train_runtime': 160.5657, 'train_samples_per_second': 2.989, 'train_steps_per_second': 0.374, 'total_flos': 706449980712960.0, 'train_loss': 1.392977664868037, 'epoch': 0.43636363636363634})

- Updates only the LoRA matrices (A, B)
- Base Phi-3 weights stay unchanged

Full Training Pipeline:
1. Load 2 examples → Forward pass → Compute loss
2. Load 2 examples → Forward pass → Accumulate gradients
3. Load 2 examples → Forward pass → Accumulate gradients  
4. Load 2 examples → Forward pass → Accumulate gradients
5. Update weights with accumulated gradients (step 1/60)
6. Repeat steps 1-5 until 60 updates complete
7. Save final model

What happens after trainer.train()?
```
Model trains for 60 steps
Saves to ./outputs/
Ready for inference or further export
```

Supervised Fine-Tuning Trainer
- Specialized trainer for fine-tuning LLMs on text data

SFTConfig = Configuration for SFTTrainer
- Just a container for all training hyperparameters
- Cleaner than passing 20 arguments directly

### Merge adapters with base model for inference

In [8]:
FastLanguageModel.for_inference(model) # Prepare for generation

messages = [
    {
        "role": "user",
        "content": "What is covered in lecture 19?"
    },
]

inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda") # Format + tokenize + add prompt

outputs = model.generate(input_ids=inputs, max_new_tokens=512, use_cache=True, temperature=0.6, do_sample=True, top_p=0.9)   # Generate response (512 tokens max, temp=0.6)

response = tokenizer.batch_decode(outputs)[0] # Convert back to text

print(response)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=512) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_m

<|user|> What is covered in lecture 19?<|end|><|assistant|> Lecture 19 (on Sunday, December 13) covers: Regression and Model Evaluation. It's held at 7:30-10 PM Egypt time.<|end|>


- Merges: W_new = W_base + (B × A)
- Now it's one unified model
- Faster inference (no separate adapter computation)

**`FastLanguageModel.for_inference(model)`**
- Switches model to **inference mode**
- Disables gradient computation (faster, less memory)
- Merges LoRA weights into base model for speed
- **Must call before generating**

**`tokenizer.apply_chat_template(...)`**
Converts chat messages into model's expected format.

**`tokenize=True`**
- Convert text → token IDs
- Output: `[1, 234, 5678, ...]` (numbers)

**`add_generation_prompt=True`**
- Adds the **assistant prompt** at the end
- Format becomes: `<|user|>What is...<|end|><|assistant|>` (prompts model to respond)
- Signals model to start generating the response

**`return_tensors="pt"`**
- Return as **PyTorch tensor** (not list)

**`.to("cuda")`**
- Move tensor to **GPU**

**`max_new_tokens=512`**
- Generate **up to 512 new tokens**
- Total output = input + 512 max

**`use_cache=True`**
- Cache attention keys/values (faster generation)
- Standard for autoregressive generation

**`temperature=0.6`**
- Controls **randomness**
- 0 = deterministic (always picks highest probability)
- 1 = very random
- 0.6 = moderately creative

**`do_sample=True`**
- Use **sampling** instead of greedy decoding
- Required when temperature > 0

**`top_p=0.9`** (nucleus sampling)
- Only sample from tokens whose **cumulative probability ≥ 90%**
- Filters out low-probability junk tokens
- Balances quality and diversity

**Output:**
```python
outputs = [[1, 234, 5678, ..., 999]]  # Token IDs (includes input + generated)
```

**`batch_decode(outputs)`**
- Convert token IDs → text
- Returns: `["<|user|>What is...<|assistant|>Lecture 19 covers..."]`

**Generation settings:**
- Temperature 0.6 = moderately creative
- Top-p 0.9 = high-quality tokens only
- Max 512 new tokens = medium-length answer

In [9]:
model.save_pretrained_gguf("gguf_model_scratch_fixed", tokenizer, quantization_method="q4_k_m", maximum_memory_usage = 0.3) # for use with llama.cpp, Ollama, LM Studio, etc.

Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/724 [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in gguf_model_scratch_fixed/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in gguf_model_scratch_fixed.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/2.65G [00:00<?, ?B/s]


Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [02:06<00:00, 63.39s/it]

Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [02:02<00:00, 61.02s/it]


Unsloth: Merge process complete. Saved to `/content/gguf_model_scratch_fixed`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['gguf_model_scratch_fixed_gguf/phi-3-mini-4k-instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['gguf_model_scratch_fixed_gguf/phi-3-mini-4k-instruct.Q4_K_M.gguf']
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model gguf_model_scratch_fixed_gguf/phi-3-mini-4k-instruct.Q4_K_M.gguf -p "why is the sky blue?"
Unsloth: Saved Ollama Modelfile to gguf_model_scratch_fixed_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f gguf_model_scratch_fixed_gguf/Modelfile


{'save_directory': 'gguf_model_scratch_fixed',
 'gguf_directory': 'gguf_model_scratch_fixed_gguf',
 'gguf_files': ['gguf_model_scratch_fixed_gguf/phi-3-mini-4k-instruct.Q4_K_M.gguf'],
 'modelfile_location': 'gguf_model_scratch_fixed_gguf/Modelfile',
 'want_full_precision': False,
 'is_vlm': False,
 'fix_bos_token': False}

maximum_memory_usage = 0.3 --> Use max 30% of available RAM during export

What happens:
- Merges LoRA weights into base model
- Converts to GGUF format
- Quantizes to 4-bit (q4_k_m)
- Saves to ./gguf_model_scratch_fixed/

Output file size estimate:
Phi-3-mini (3.8B parameters):
- Original (FP16): ~7.6 GB
- After q4_k_m: ~2.1 GB

Why GGUF?
- ✅ Run on CPU (no GPU needed)
- ✅ Works with llama.cpp, Ollama, LM Studio
- ✅ Much smaller file size
- ✅ Fast inference on consumer

```
Before get_peft_model():
┌─────────────────┐
│  Base Model     │  ← All weights frozen
│  (Phi-3)        │
└─────────────────┘

After get_peft_model():
┌─────────────────┐
│  Base Model     │  ← Frozen (not trained)
│  (Phi-3)        │
└─────────────────┘
        +
┌─────────────────┐
│ LoRA Adapters   │  ← Trainable (A, B matrices)
│ (rank 64)       │  ← Only these get updated
└─────────────────┘

After training:
┌─────────────────┐
│  Base Model     │  ← Still original
└─────────────────┘
        +
┌─────────────────┐
│ Trained LoRA    │  ← Updated with your data
└─────────────────┘

After for_inference():
┌─────────────────┐
│  Merged Model   │  ← Base + LoRA combined
│  (One model)    │
└─────────────────┘
```
- ✅ Define LoRA adapters (get_peft_model)
- ✅ Train adapter weights on your data (trainer.train())
- ✅ Merge adapters with base model for inference (for_inference())